## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [24]:
!pip install seaborn pandas

import seaborn as sns

titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
860,0,3,male,41.0,2,0,14.1083,S,Third,man,True,NaN,Southampton,no,False
716,1,1,female,38.0,0,0,227.5250,C,First,woman,False,C,Cherbourg,yes,True
606,0,3,male,30.0,0,0,7.8958,S,Third,man,True,NaN,Southampton,no,True
52,1,1,female,49.0,1,0,76.7292,C,First,woman,False,D,Cherbourg,yes,False
656,0,3,male,NaN,0,0,7.8958,S,Third,man,True,NaN,Southampton,no,True


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [25]:
missing_data = titanic_data.isnull().sum()
print(missing_data)

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [26]:
limit_cols = len(titanic_data) / 2
titanic_data = titanic_data.dropna(axis=1, thresh=limit_cols)
limit_rows = len(titanic_data.columns) / 2
titanic_data = titanic_data.dropna(axis=0, thresh=limit_rows)

### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [27]:
med_man = round(titanic_data[titanic_data["who"] == "man"]["age"].median())
med_woman = round(titanic_data[titanic_data["who"] == "woman"]["age"].median())
med_child = round(titanic_data[titanic_data["who"] == "child"]["age"].median())
titanic_data.loc[(titanic_data["who"] == "man") & (titanic_data["age"].isnull()), "age"] = med_man
titanic_data.loc[(titanic_data["who"] == "woman") & (titanic_data["age"].isnull()), "age"] = (
    med_woman
)
titanic_data.loc[(titanic_data["who"] == "child") & (titanic_data["age"].isnull()), "age"] = (
    med_child
)

### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [28]:
total_columns = len(titanic_data.columns)
titanic_data = titanic_data.dropna(thresh=total_columns - 1)

### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [29]:
cnt_city = titanic_data["embark_town"].value_counts()
top_city = cnt_city.idxmax()
print(top_city)

Southampton


### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [30]:
survive_percent = (titanic_data["survived"].mean() * 100).round(2)

print(survive_percent)

38.25


### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [31]:
survivors_for_city = titanic_data.groupby("embark_town")["survived"].sum()
print(survivors_for_city)

embark_town
Cherbourg       93
Queenstown      30
Southampton    217
Name: survived, dtype: int64


### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [32]:
class_survival = (titanic_data.groupby("class")["survived"].mean() * 100).round(2)
print(class_survival)

class
First     62.62
Second    47.28
Third     24.24
Name: survived, dtype: float64


### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [33]:
rich_passengers = titanic_data[titanic_data["fare"] >= 100]
rich_survival = (rich_passengers["survived"].mean() * 100).round(2)
print(f"{rich_survival}%")

73.58%


### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [34]:
lonely_children = titanic_data[(titanic_data["who"] == "child") & (titanic_data["alone"] == True)]
print(len(lonely_children))

6


Какие выводы вы можете сделать о выживших пассажирах Титаника? 